# 📊 Comprehensive Property Analysis

**s-CGCNN Version 0.1 - Notebook 3**

Deep dive into AlₓGa₁₋ₓAs properties across all 41 compositions.

---

## Setup

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Imports successful")

## 1. Load All Generated Data

In [ ]:
# Load metadata
metadata_dir = Path("../data/structures/metadata")
metadata_files = sorted(metadata_dir.glob("AlGaAs_x_*.json"))

print(f"Loading {len(metadata_files)} metadata files...\n")

if len(metadata_files) == 0:
    print("❌ ERROR: No metadata found!")
    print("Please run notebook 02_Structure_Interpolation_Demo.ipynb first")
else:
    # Load all data
    all_properties = []
    for f in metadata_files:
        with open(f, 'r') as file:
            data = json.load(file)
            props = data['properties'].copy()
            props['x'] = data['x_value']
            props['composition'] = data['composition']
            props['lattice_a'] = data['lattice_parameters']['a']
            props['volume'] = data['lattice_parameters']['volume']
            all_properties.append(props)
    
    df = pd.DataFrame(all_properties)
    print(f"✓ Loaded {len(df)} compositions")
    print(f"  x range: {df['x'].min():.3f} to {df['x'].max():.3f}")
    print(f"  Total properties: {len(df.columns)}")

## 2. Property Overview

In [ ]:
# Display first few rows
display_cols = ['x', 'composition', 'lattice_a', 'band_gap', 'band_gap_type', 
                'density', 'thermal_conductivity']
print("Sample data (first 10 compositions):\n")
print(df[display_cols].head(10).to_string(index=False))

In [ ]:
# Statistical summary
numeric_cols = df.select_dtypes(include=[np.number]).columns
print("\nStatistical Summary:\n")
print(df[numeric_cols].describe().round(3))

## 3. Electronic Properties Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Band gap with type coloring
direct = df[df['band_gap_type'] == 'direct']
indirect = df[df['band_gap_type'] == 'indirect']

axes[0, 0].scatter(direct['x'], direct['band_gap'], c='blue', s=50, label='Direct', alpha=0.7)
axes[0, 0].scatter(indirect['x'], indirect['band_gap'], c='red', s=50, label='Indirect', alpha=0.7)
axes[0, 0].axvline(x=0.45, color='gray', linestyle='--', alpha=0.5, label='Crossover')
axes[0, 0].set_xlabel('Al fraction (x)', fontsize=11)
axes[0, 0].set_ylabel('Band gap (eV)', fontsize=11)
axes[0, 0].set_title('Band Gap with Direct/Indirect Transition', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Electron affinity
axes[0, 1].plot(df['x'], df['electron_affinity'], 'o-', color='purple', markersize=5)
axes[0, 1].set_xlabel('Al fraction (x)', fontsize=11)
axes[0, 1].set_ylabel('Electron Affinity (eV)', fontsize=11)
axes[0, 1].set_title('Electron Affinity Evolution', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Dielectric constants
axes[1, 0].plot(df['x'], df['static_dielectric'], 'o-', label='Static (ε₀)', markersize=5)
axes[1, 0].plot(df['x'], df['optical_dielectric'], 's-', label='Optical (ε∞)', markersize=5)
axes[1, 0].set_xlabel('Al fraction (x)', fontsize=11)
axes[1, 0].set_ylabel('Dielectric Constant', fontsize=11)
axes[1, 0].set_title('Dielectric Constants', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Effective masses
axes[1, 1].plot(df['x'], df['electron_effective_mass'], 'o-', label='Electron', markersize=5)
axes[1, 1].plot(df['x'], df['hole_effective_mass_heavy'], 's-', label='Heavy hole', markersize=5)
axes[1, 1].set_xlabel('Al fraction (x)', fontsize=11)
axes[1, 1].set_ylabel('Effective Mass (m₀)', fontsize=11)
axes[1, 1].set_title('Carrier Effective Masses', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Mechanical Properties

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elastic constants
axes[0].plot(df['x'], df['elastic_c11'], 'o-', label='C₁₁', markersize=5)
axes[0].plot(df['x'], df['elastic_c12'], 's-', label='C₁₂', markersize=5)
axes[0].plot(df['x'], df['elastic_c44'], '^-', label='C₄₄', markersize=5)
axes[0].set_xlabel('Al fraction (x)', fontsize=11)
axes[0].set_ylabel('Elastic Constants (GPa)', fontsize=11)
axes[0].set_title('Elastic Constants', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bulk and shear modulus
axes[1].plot(df['x'], df['bulk_modulus'], 'o-', label='Bulk modulus', markersize=6)
axes[1].plot(df['x'], df['shear_modulus'], 's-', label='Shear modulus', markersize=6)
axes[1].set_xlabel('Al fraction (x)', fontsize=11)
axes[1].set_ylabel('Modulus (GPa)', fontsize=11)
axes[1].set_title('Bulk & Shear Modulus', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Thermal Properties

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Thermal conductivity
axes[0, 0].plot(df['x'], df['thermal_conductivity'], 'o-', color='red', markersize=6)
axes[0, 0].set_xlabel('Al fraction (x)', fontsize=11)
axes[0, 0].set_ylabel('Thermal Conductivity (W/m·K)', fontsize=11)
axes[0, 0].set_title('Thermal Conductivity (Critical for Heat Management)', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Thermal expansion
axes[0, 1].plot(df['x'], df['thermal_expansion']*1e6, 'o-', color='orange', markersize=6)
axes[0, 1].set_xlabel('Al fraction (x)', fontsize=11)
axes[0, 1].set_ylabel('Thermal Expansion (10⁻⁶/K)', fontsize=11)
axes[0, 1].set_title('Thermal Expansion Coefficient', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Specific heat
axes[1, 0].plot(df['x'], df['specific_heat'], 'o-', color='green', markersize=6)
axes[1, 0].set_xlabel('Al fraction (x)', fontsize=11)
axes[1, 0].set_ylabel('Specific Heat (J/kg·K)', fontsize=11)
axes[1, 0].set_title('Specific Heat Capacity', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Debye temperature
axes[1, 1].plot(df['x'], df['debye_temperature'], 'o-', color='purple', markersize=6)
axes[1, 1].set_xlabel('Al fraction (x)', fontsize=11)
axes[1, 1].set_ylabel('Debye Temperature (K)', fontsize=11)
axes[1, 1].set_title('Debye Temperature', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Transport Properties (Mobility)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(df['x'], df['electron_mobility'], 'o-', label='Electron', markersize=6, linewidth=2)
ax.semilogy(df['x'], df['hole_mobility'], 's-', label='Hole', markersize=6, linewidth=2)
ax.axvline(x=0.45, color='gray', linestyle='--', alpha=0.5, label='Direct/Indirect crossover')
ax.set_xlabel('Al fraction (x)', fontsize=12)
ax.set_ylabel('Mobility (cm²/V·s, log scale)', fontsize=12)
ax.set_title('Carrier Mobility Evolution (Critical for Device Performance)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("Key observations:")
print(f"  • GaAs electron mobility (x=0): {df[df['x']==0]['electron_mobility'].values[0]:.0f} cm²/V·s")
print(f"  • AlAs electron mobility (x=1): {df[df['x']==1]['electron_mobility'].values[0]:.0f} cm²/V·s")
print(f"  • Significant drop after direct→indirect transition")

## 7. Correlation Matrix (Heatmap)

In [ ]:
# Select key properties for correlation
corr_cols = ['x', 'lattice_a', 'band_gap', 'density', 'electron_affinity',
            'static_dielectric', 'bulk_modulus', 'thermal_conductivity',
            'electron_effective_mass', 'electron_mobility']

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
           center=0, square=True, linewidths=1)
plt.title('Property Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 8. Device-Relevant Property Combinations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Band gap vs Mobility (HEMT relevance)
axes[0].scatter(df['band_gap'], df['electron_mobility'], 
               c=df['x'], cmap='viridis', s=80, alpha=0.7)
axes[0].set_xlabel('Band Gap (eV)', fontsize=11)
axes[0].set_ylabel('Electron Mobility (cm²/V·s)', fontsize=11)
axes[0].set_title('Band Gap vs Mobility (HEMT Performance)', fontweight='bold')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)
cbar1 = plt.colorbar(axes[0].collections[0], ax=axes[0])
cbar1.set_label('Al fraction (x)', fontsize=10)

# Thermal conductivity vs Band gap (Heat management)
axes[1].scatter(df['band_gap'], df['thermal_conductivity'], 
               c=df['x'], cmap='plasma', s=80, alpha=0.7)
axes[1].set_xlabel('Band Gap (eV)', fontsize=11)
axes[1].set_ylabel('Thermal Conductivity (W/m·K)', fontsize=11)
axes[1].set_title('Band Gap vs Thermal Conductivity', fontweight='bold')
axes[1].grid(True, alpha=0.3)
cbar2 = plt.colorbar(axes[1].collections[0], ax=axes[1])
cbar2.set_label('Al fraction (x)', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Identify Optimal Compositions for Devices

In [ ]:
print("="*70)
print("OPTIMAL COMPOSITIONS FOR DEVICE APPLICATIONS")
print("="*70)

# HEMT: High mobility + moderate band gap
print("\n1. High Electron Mobility Transistor (HEMT):")
print("   Requirements: High electron mobility, direct gap")
hemt_candidates = df[(df['electron_mobility'] > 5000) & (df['band_gap_type'] == 'direct')]
best_hemt = hemt_candidates.nlargest(3, 'electron_mobility')
print(f"   Top 3 candidates:")
for _, row in best_hemt.iterrows():
    print(f"     x={row['x']:.3f}: mobility={row['electron_mobility']:.0f} cm²/V·s, Eg={row['band_gap']:.3f} eV")

# Laser diode: Direct gap in visible/IR range
print("\n2. Laser Diode / LED:")
print("   Requirements: Direct band gap 1.4-2.2 eV")
laser_candidates = df[(df['band_gap'] >= 1.4) & (df['band_gap'] <= 2.2) & (df['band_gap_type'] == 'direct')]
print(f"   Candidates: {len(laser_candidates)}")
print(f"   x range: {laser_candidates['x'].min():.3f} - {laser_candidates['x'].max():.3f}")
print(f"   Band gap range: {laser_candidates['band_gap'].min():.3f} - {laser_candidates['band_gap'].max():.3f} eV")

# Solar cell: Optimal band gap ~1.4 eV
print("\n3. Solar Cell:")
print("   Requirements: Band gap ≈ 1.42 eV (optimal for solar spectrum)")
solar_candidates = df[(df['band_gap'] >= 1.35) & (df['band_gap'] <= 1.50)]
best_solar = solar_candidates.iloc[(solar_candidates['band_gap'] - 1.42).abs().argsort()[:3]]
print(f"   Top 3 candidates:")
for _, row in best_solar.iterrows():
    print(f"     x={row['x']:.3f}: Eg={row['band_gap']:.3f} eV, type={row['band_gap_type']}")

# Microprocessor: Good mobility + thermal management
print("\n4. Microprocessor Logic:")
print("   Requirements: High mobility + high thermal conductivity")
proc_candidates = df[(df['electron_mobility'] > 4000) & (df['thermal_conductivity'] > 30)]
best_proc = proc_candidates.nlargest(3, 'electron_mobility')
print(f"   Top 3 candidates:")
for _, row in best_proc.iterrows():
    print(f"     x={row['x']:.3f}: mobility={row['electron_mobility']:.0f} cm²/V·s, κ={row['thermal_conductivity']:.1f} W/m·K")

print("\n" + "="*70)

## 10. Export Analysis Results

In [ ]:
# Save full dataset as CSV
output_file = Path("../results/algaas_properties_analysis.csv")
output_file.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_file, index=False)
print(f"✓ Full dataset saved to: {output_file}")

# Save summary statistics
summary_file = Path("../results/property_summary_statistics.csv")
df.describe().to_csv(summary_file)
print(f"✓ Summary statistics saved to: {summary_file}")

## Summary

In [ ]:
print("="*70)
print("PROPERTY ANALYSIS COMPLETE")
print("="*70)
print(f"\n✓ Analyzed {len(df)} compositions")
print(f"✓ Explored {len(df.columns)} properties")
print(f"✓ Identified optimal compositions for 4 device types")
print(f"\n→ Next steps:")
print(f"  1. Move to Version 0.2 for visualization")
print(f"  2. Prepare data for graph neural network training")
print(f"  3. Develop device recommendation algorithm")
print("\n" + "="*70)

---

## ✅ Analysis Checklist

- [x] All data loaded successfully
- [x] Electronic properties analyzed
- [x] Mechanical properties examined
- [x] Thermal properties evaluated
- [x] Transport properties assessed
- [x] Property correlations identified
- [x] Device-relevant compositions found
- [x] Results exported

**Status:** Complete ✓